# AEGIS-RF — гібридна байєсівська просторова ідентифікація Wi-Fi у КІІ
## Adversary-rEsistant Geolocation & Integrity for wi-fi Signals

Прикладний демонстраційний програмний проєкт **за Розділом 1** дисертаційного
дослідження: просторова ідентифікація джерел IEEE 802.11 у критичній інформаційній
інфраструктурі (КІІ) зі злиттям **RSSI-радіокарти** та **FTM/RTT-дальнометрії**,
**робастністю** до навмисних маніпуляцій (evil twin, deceptive ranging, deauth) і
**доказовою атрибуцією** (Spatial Attribution Record) для інтеграції із SOC/SIEM/SOAR.

> **Демонстраційні дані.** Усі числа характеризують синтетичну модель, параметри якої
> задані автором, а не реальні вимірювання. Проєкт ілюструє методологію та перевіряє
> відтворюваність обчислень.

**Реалізовані компоненти розділу:**
> 1.1 моделі загроз (rogue AP / evil twin / deauth) · 1.2 RSSI-fingerprinting і
> байєсівська локалізація · 1.3 FTM/RTT-дальнометрія · 1.4 гібридне злиття
> правдоподібностей і робастний інференс · 1.5 доказова атрибуція та SOC/SIEM.

## 0. Налаштування / Setup

Ноутбук імпортує пакет `src/` та фіксує зерно `SEED = 80211`.

**▶ Запуск у Colab:**
```
!git clone https://github.com/omega2417/bnt.git
%cd bnt/aegis-rf
```
далі *Runtime → Run all*.

In [ ]:
# ============================================================
# Налаштування середовища / Environment setup
# ============================================================
import os, sys, json, hashlib, warnings

def _find_root(depth=4):
    p = os.getcwd()
    for _ in range(depth):
        if os.path.isfile(os.path.join(p, "src", "pipeline.py")):
            return p
        p = os.path.dirname(p)
    return None

ROOT = _find_root() or os.getcwd()
sys.path.insert(0, ROOT); os.chdir(ROOT)

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from src import pipeline as P
from src import make_figures as figs

warnings.filterwarnings("ignore")
figs.setup_matplotlib()
for d in ("results", "figures"):
    os.makedirs(d, exist_ok=True)

print("Project root:", ROOT)
print("SEED:", P.SEED, "| events:", P.N_EVENTS, "| scenarios:", P.SCENARIOS)
print("NumPy:", np.__version__)

## 1. Середовище КІІ / CII environment

Синтетичне приміщення $30\times24$ м із шістьма точками доступу IEEE 802.11
(частина підтримує FTM/RTT), виділеною **критичною зоною** (напр. серверна) та
регулярною RP-сіткою радіокарти. Джерела сигналу (події) рівномірно розподілені,
близько 45 % — усередині критичної зони.

In [ ]:
exp = P.run_experiment()
figs.fig_environment(exp, save="figures/fig1_environment.png"); plt.show()

## 2. Локалізація та злиття модальностей / Localization & fusion

Байєсівська локалізація на RP-сітці: правдоподібність позиції — добуток гауссових
членів по доступних AP (RSSI-радіокарта та FTM-дальнометрія). Порівнюємо чотири методи:

* **RSSI** — лише радіокарта;
* **FTM/RTT** — лише часова дальнометрія;
* **Наївне злиття** — зважений добуток правдоподібностей;
* **Робастне злиття** — те саме з ітеративним відсіюванням AP-викидів (стійкість
  до evil twin / deceptive ranging).

Проганяємо всі методи на 4 сценаріях (без атак, evil twin, deceptive ranging, deauth)
і зводимо метрики в таблицю.

In [ ]:
exp, results, table = P.run()
cols = ["scenario","method","median_err_m","rmse_m","p90_err_m","zone_acc","crit_pd","crit_far"]
table.to_csv("results/metrics.csv", index=False)
table[cols].round(4)

### Інтерпретація

* **Без атак** гібридне злиття точніше за окремі модальності (менша медіанна похибка).
* **Evil twin** підриває RSSI-локалізацію; FTM лишається чистим, а робастне злиття
  відсіює підроблений AP і зберігає точність.
* **Deceptive ranging** «підтягує» наївне злиття (FTM-канал скомпрометовано), але
  робастний метод відновлює позицію.
* **Deauth** — плавна деградація: злиття зберігає працездатність за втрати одного AP.

In [ ]:
pivot = table.pivot(index="scenario", columns="method", values="median_err_m")[
    ["rssi","ftm","fusion","robust"]].reindex(P.SCENARIOS)
display(pivot.round(3))
figs.fig_error_bars(table, save="figures/fig3_error_bars.png"); plt.show()

## 3. Розподіл похибки / Error CDF

Емпіричні функції розподілу похибки локалізації (сценарій без атак): робастне та
наївне злиття домінують над окремими модальностями на всьому діапазоні.

In [ ]:
figs.fig_error_cdf(exp, scenario="clean", save="figures/fig2_error_cdf.png"); plt.show()

## 4. Зонова атрибуція / Zone attribution

Апостеріорна ймовірність перебування джерела в критичній зоні → рішення тривоги.
**Pd** — виявлення істинно-критичних подій, **FAR** — хибні тривоги на дозволених
позиціях. Робастне злиття утримує Pd під атаками, коли скомпрометована модальність
завалює наївні методи.

In [ ]:
figs.fig_attribution(table, save="figures/fig4_attribution.png"); plt.show()

## 5. Просторовий ефект атаки / Spatial effect of the attack

Лінії з'єднують істинну позицію з оцінкою. За «evil twin» наївне злиття системно
зміщує оцінки до підробленого AP; робастний метод відсіює викид і повертає їх на місце.

In [ ]:
figs.fig_attack_shift(exp, scenario="evil_twin", save="figures/fig5_attack_shift.png"); plt.show()

## 6. Spatial Attribution Record (SAR) для SOC/SIEM

Мінімальна одиниця доказу (п. 1.5.3): оцінена позиція, невизначеність, апостеріорна
ймовірність зони, використані модальності, ознаки цілісності (`integrity_flags`) та
провенанс. Формат — JSON по об'єкту на рядок, придатний для SIEM.

In [ ]:
from src.attribution import build_records, to_siem_json, integrity_flags
from src.localization import localize

obs = exp.scenarios["evil_twin"]
res = localize(obs, exp.radiomap, "robust")
flags = integrity_flags(obs, exp.radiomap)
records = build_records(obs, res, exp.radiomap, scenario="evil_twin", method="robust",
                        seed=P.SEED, integrity_flags=flags, indices=range(len(obs.pos)))
with open("results/spatial_attribution_records.jsonl", "w", encoding="utf-8") as fh:
    fh.write(to_siem_json(records))
print(f"Сформовано {len(records)} SAR-записів | подій з ознакою неузгодженості: {int(flags.sum())}")
print("\nПриклад запису:")
import json; print(json.dumps(records[0], ensure_ascii=False, indent=2))

## 7. Артефакти та контрольні суми / Artifacts & checksums

Таблиці — в `results/`, рисунки (320 dpi) — у `figures/`, доказові записи — у
`results/spatial_attribution_records.jsonl`. SHA-256 фіксують точний вміст для депозиту.

In [ ]:
manifest = {}
for root in ("results", "figures"):
    for f in sorted(os.listdir(root)):
        p = os.path.join(root, f)
        manifest[p] = hashlib.sha256(open(p, "rb").read()).hexdigest()[:16]
with open("results/MANIFEST_sha256.json", "w") as fh:
    json.dump(manifest, fh, indent=2)
for p, h in manifest.items():
    print(f"{h}  {p}  ({os.path.getsize(p)/1024:.1f} КБ)")

## 8. Обмеження / Limitations

1. **Синтетичні дані.** Модель загасання, похибки FTM, частки NLOS та відмов задані
   параметрично, а не з польових вимірювань.
2. **Спрощена геометрія.** Один поверх, прямокутна критична зона, без стін/матеріалів
   і повного багатопроменевого моделювання.
3. **Модель супротивника** обмежена трьома класами атак на один AP; коаліційні та
   адаптивні атаки не розглядаються.
4. **Немає прямого емпіричного порівняння** з польовими системами.
5. Наведені числа **не можна використовувати** для сертифікації чи проєктування
   реальних систем захисту КІІ без валідації на реальних даних.

Ліцензія: MIT. Цитування — `CITATION.cff`. Прикладний проєкт за Розділом 1.